<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.1-document-ai/practice/GCP_Capstone_4.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 4.1 — Document AI — OCR, Layout Parser, Form Parser

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the Document AI + Firestore SDKs, authenticate with Application Default Credentials (no API keys), and paste your project + processor IDs. Create the three processors (OCR, Layout Parser, Form Parser) in the Cloud Console first, then paste their IDs below.

In [ ]:
!pip install -q google-cloud-documentai google-cloud-documentai-toolbox google-cloud-firestore reportlab
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'us'  # Document AI location: 'us' or 'eu'

# --- 1. Generate a sample test.pdf so you don't have to upload one ---
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
_c = canvas.Canvas('test.pdf', pagesize=letter)
_c.setFont('Helvetica-Bold', 16); _c.drawString(72, 720, 'DocuMind AI - Sample Invoice')
_c.setFont('Helvetica', 11)
for _i, _line in enumerate([
        'Invoice Number: INV-2026-0042', 'Date: 2026-08-31',
        'Bill To: Acme Corp, Hyderabad, Telangana',
        'Item: GCP GenAI Capstone Training       Amount: Rs 85,000',
        'Item: Vertex AI Consulting (20 hrs)     Amount: Rs 1,20,000',
        'Total: Rs 2,05,000',
        'GSTIN: 36ABCDE1234F1Z5    Payment due in 30 days.']):
    _c.drawString(72, 690 - _i * 22, _line)
_c.showPage(); _c.save()
print('Wrote test.pdf')

# --- 1b. A form.pdf (labelled fields + a table) for the Form Parser (Cell 4) ---
_f = canvas.Canvas('form.pdf', pagesize=letter)
_f.setFont('Helvetica-Bold', 15); _f.drawString(72, 730, 'DocuMind AI - Onboarding Form')
_f.setFont('Helvetica', 11)
for _i, _line in enumerate([
        'Full Name: Priya Sharma', 'Employee ID: EMP-2026-118',
        'Department: Machine Learning', 'Date of Joining: 2026-09-01',
        'Location: Hyderabad', 'Email: priya.sharma@acme.in',
        'Manager: Rahul Verma']):
    _f.drawString(72, 700 - _i * 24, _line)
_f.setFont('Helvetica-Bold', 11); _f.drawString(72, 510, 'Equipment Issued')
_y = 490
for _r, (_a, _b) in enumerate([('Item', 'Serial'), ('Laptop', 'LT-9921'),
                               ('Monitor', 'MN-4415'), ('Access Card', 'AC-7788')]):
    _f.setFont('Helvetica-Bold' if _r == 0 else 'Helvetica', 10)
    _f.drawString(80, _y, _a); _f.drawString(260, _y, _b)
    _f.line(72, _y - 5, 400, _y - 5)
    _y -= 22
_f.showPage(); _f.save()
print('Wrote form.pdf')

# --- 2. Document AI processors: created once if missing (or paste your own IDs) ---
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
_da = documentai.DocumentProcessorServiceClient(
    client_options=ClientOptions(api_endpoint=f'{LOCATION}-documentai.googleapis.com'))
_parent = _da.common_location_path(PROJECT_ID, LOCATION)

def _ensure_processor(display_name, type_):
    # reuse an existing processor of this type, else create one (needs the Document
    # AI API enabled + documentai.processors.create; else create in the Console and
    # paste the ID here).
    for p in _da.list_processors(parent=_parent):
        if p.type_ == type_:
            return p.name.split('/')[-1]
    p = _da.create_processor(parent=_parent,
        processor=documentai.Processor(type_=type_, display_name=display_name))
    return p.name.split('/')[-1]

OCR_ID    = _ensure_processor('documind-ocr',    'OCR_PROCESSOR')
LAYOUT_ID = _ensure_processor('documind-layout', 'LAYOUT_PARSER_PROCESSOR')
FORM_ID   = _ensure_processor('documind-form',   'FORM_PARSER_PROCESSOR')
print(f'Processors ready: OCR={OCR_ID}  LAYOUT={LAYOUT_ID}  FORM={FORM_ID}')

# --- 3. Firestore (default) database for Cell 5 (created once if missing) ---
import subprocess
FIRESTORE_LOCATION = 'asia-south1'  # region for the (default) DB; PERMANENT once created
if '(default)' not in subprocess.run(
        ['gcloud', 'firestore', 'databases', 'list', '--project', PROJECT_ID, '--format=value(name)'],
        capture_output=True, text=True).stdout:
    print(f'Creating Firestore (default) database in {FIRESTORE_LOCATION} (one-time)...')
    subprocess.run(['gcloud', 'firestore', 'databases', 'create',
                    '--location=' + FIRESTORE_LOCATION, '--project', PROJECT_ID], check=False)
else:
    print('Firestore (default) database ready.')

USD_INR = 85  # for any cost display

### Universal processing function

Every exercise routes through this one helper: build a `ProcessRequest` with a `RawDocument`, call `process_document()`, return the parsed `Document`. The regional `api_endpoint` must match `LOCATION`.

In [ ]:
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

def process_document(project_id, location, processor_id, file_path,
                     mime_type='application/pdf', process_options=None):
    client = documentai.DocumentProcessorServiceClient(
        client_options=ClientOptions(
            api_endpoint=f'{location}-documentai.googleapis.com'))
    name = client.processor_path(project_id, location, processor_id)
    with open(file_path, 'rb') as f:
        content = f.read()
    request = documentai.ProcessRequest(
        name=name,
        raw_document=documentai.RawDocument(content=content, mime_type=mime_type),
        process_options=process_options)
    return client.process_document(request=request).document

print('process_document() ready')

## Exercise 1: First OCR Call

**Difficulty:** Easy

Create an OCR processor. Process a 1-page PDF. Print full text and token count.

1. Create processor in Console or via SDK
2. Call process_document()
3. Print doc.text and count doc.pages[0].tokens

In [ ]:
# Upload a test PDF to Colab first:
# from google.colab import files
# uploaded = files.upload()   # save as 'test.pdf'

options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True))

doc = process_document(PROJECT_ID, LOCATION, OCR_ID, 'test.pdf',
                       process_options=options)

print(f'Full text ({len(doc.text)} chars):')
print(doc.text[:500])

token_count = len(doc.pages[0].tokens)
print(f'\nPage 1 token count: {token_count}')

## Exercise 2: Page-Level Details

**Difficulty:** Easy

Process a multi-page PDF. Print languages, paragraph count, quality per page.

1. Use enable_image_quality_scores=True
2. Loop through doc.pages
3. Print detected_languages and quality_score

In [ ]:
options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True,
        enable_image_quality_scores=True))

doc = process_document(PROJECT_ID, LOCATION, OCR_ID, 'test.pdf',
                       process_options=options)

print(f'Pages: {len(doc.pages)}')
def _anchor_text(document, text_anchor):
    # Online process() leaves text_anchor.content empty; resolve text from document.text
    if not text_anchor.text_segments:
        return (text_anchor.content or '').strip()
    return ''.join(document.text[int(s.start_index):int(s.end_index)]
                   for s in text_anchor.text_segments).strip()

for page in doc.pages:
    langs = [(l.language_code, f'{l.confidence:.0%}') for l in page.detected_languages]
    quality = page.image_quality_scores.quality_score
    print(f'  Page {page.page_number}: {len(page.paragraphs)} paragraphs, '
          f'langs={langs}, quality={quality:.2f}')

## Exercise 3: Hindi OCR

**Difficulty:** Easy

Process a Hindi document with language_hints=["hi"]. Verify Devanagari extraction.

1. Set hints in OcrConfig
2. Process Hindi PDF/image
3. Verify Devanagari script in output

In [ ]:
# Upload a Hindi PDF/image as 'hindi.pdf' first.
options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True,
        hints=documentai.OcrConfig.Hints(language_hints=['hi'])))

doc = process_document(PROJECT_ID, LOCATION, OCR_ID, 'hindi.pdf',
                       process_options=options)

print('Extracted text (first 500 chars):')
print(doc.text[:500])

# Verify Devanagari (Unicode block U+0900–U+097F) is present
has_devanagari = any('ऀ' <= ch <= 'ॿ' for ch in doc.text)
print(f'\nDevanagari script present: {has_devanagari}')
for page in doc.pages:
    langs = [(l.language_code, f'{l.confidence:.0%}') for l in page.detected_languages]
    print(f'  Page {page.page_number} detected_languages: {langs}')

## Exercise 4: Layout Parser Chunking

**Difficulty:** Medium

Process a research paper with Layout Parser. Print all chunks with ancestor headings.

1. Set include_ancestor_headings=True, chunk_size=1024
2. Process PDF with Layout Parser
3. Print each chunk with its heading context

In [ ]:
options = documentai.ProcessOptions(
    layout_config=documentai.ProcessOptions.LayoutConfig(
        enable_table_annotation=True,
        enable_image_annotation=True,
        chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
            chunk_size=1024,
            include_ancestor_headings=True)))

doc = process_document(PROJECT_ID, LOCATION, LAYOUT_ID, 'test.pdf',
                       process_options=options)

print(f'Chunks: {len(doc.chunked_document.chunks)}')
for i, chunk in enumerate(doc.chunked_document.chunks[:5]):
    print(f'\n--- Chunk {i} ({chunk.chunk_id}) ---')
    print(f'Pages: {chunk.page_span.page_start}-{chunk.page_span.page_end}')
    print(f'Content:\n{chunk.content[:200]}...')

## Exercise 5: Form Parser KVP

**Difficulty:** Medium

Process an invoice or form PDF. Extract key-value pairs and tables.

1. Process with Form Parser
2. Loop page.form_fields for KVPs
3. Loop page.tables for structured data

In [ ]:
# Upload an invoice/form PDF as 'form.pdf' first.
doc = process_document(PROJECT_ID, LOCATION, FORM_ID, 'form.pdf')

for page in doc.pages:
    print(f'\n=== Page {page.page_number} ===')
    for field in page.form_fields:
        key = _anchor_text(doc, field.field_name.text_anchor)
        val = _anchor_text(doc, field.field_value.text_anchor)
        print(f'  {key}: {val} ({field.field_value.confidence:.0%})')
    for idx, table in enumerate(page.tables):
        print(f'\n  Table {idx}:')
        for row in table.body_rows:
            cells = [_anchor_text(doc, c.layout.text_anchor) for c in row.cells]
            print(f'    {cells}')

## Exercise 6: Quality Gate Pipeline

**Difficulty:** Medium

OCR with quality scores. Filter pages below 0.5. Only chunk high-quality pages.

1. Enable quality scores in OCR
2. Filter pages where quality_score >= 0.5
3. Only include high-quality page text in chunks

In [ ]:
# Step 1 — OCR with quality scores to find the good pages.
ocr_options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True,
        enable_image_quality_scores=True))

ocr_doc = process_document(PROJECT_ID, LOCATION, OCR_ID, 'test.pdf',
                           process_options=ocr_options)

QUALITY_THRESHOLD = 0.5
good_pages, bad_pages = [], []
for page in ocr_doc.pages:
    score = page.image_quality_scores.quality_score
    (good_pages if score >= QUALITY_THRESHOLD else bad_pages).append(
        (page.page_number, score))

print(f'High-quality pages (>= {QUALITY_THRESHOLD}): {good_pages}')
print(f'Rejected pages: {bad_pages}')

# Step 2 — only chunk if the document cleared the gate. Chunking is
# document-level, so we gate the whole doc on having usable pages and
# keep only chunks that fall entirely within high-quality page ranges.
good_page_numbers = {pn for pn, _ in good_pages}
if good_page_numbers:
    layout_options = documentai.ProcessOptions(
        layout_config=documentai.ProcessOptions.LayoutConfig(
            chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
                chunk_size=1024, include_ancestor_headings=True)))
    layout_doc = process_document(PROJECT_ID, LOCATION, LAYOUT_ID, 'test.pdf',
                                  process_options=layout_options)
    kept = [c for c in layout_doc.chunked_document.chunks
            if all(p in good_page_numbers
                   for p in range(c.page_span.page_start, c.page_span.page_end + 1))]
    print(f'\nKept {len(kept)} / {len(layout_doc.chunked_document.chunks)} chunks '
          f'after quality gate')
else:
    print('\nNo pages cleared the quality gate — nothing chunked.')

## Exercise 7: Chunks to Firestore

**Difficulty:** Challenge

Layout Parser chunks into Firestore with full metadata. Verify in Console.

1. Process with Layout Parser
2. Store each chunk with source, pages, chunk_id
3. Verify in Firestore Console

In [ ]:
from google.cloud import firestore

def store_chunks(document, source_file):
    db = firestore.Client(project=PROJECT_ID)
    batch = db.batch()
    for chunk in document.chunked_document.chunks:
        ref = db.collection('rag_chunks').document(chunk.chunk_id)
        batch.set(ref, {
            'source_file': source_file,
            'content': chunk.content,
            'page_start': chunk.page_span.page_start,
            'page_end': chunk.page_span.page_end,
            'processed_at': firestore.SERVER_TIMESTAMP})
    batch.commit()
    print(f'Stored {len(document.chunked_document.chunks)} chunks')

# Reuse the Layout-Parser output from Exercise 4 (variable `doc`), or re-run:
options = documentai.ProcessOptions(
    layout_config=documentai.ProcessOptions.LayoutConfig(
        chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
            chunk_size=1024, include_ancestor_headings=True)))
doc = process_document(PROJECT_ID, LOCATION, LAYOUT_ID, 'test.pdf',
                       process_options=options)

store_chunks(doc, 'test.pdf')
# Now open Firestore Console → 'rag_chunks' collection to verify.

## Exercise 8: DocumentIngester Module

**Difficulty:** Challenge

Build complete DocumentIngester class with process() and ingest_for_rag().

1. Implement process() for any processor
2. Implement ingest_for_rag() with Layout + Firestore
3. Test end-to-end with a research paper

In [ ]:
class DocumentIngester:
    def __init__(self, project_id, location='us'):
        self.project_id = project_id
        self.location = location
        self.client = documentai.DocumentProcessorServiceClient(
            client_options=ClientOptions(
                api_endpoint=f'{location}-documentai.googleapis.com'))
        self.db = firestore.Client(project=self.project_id)

    def process(self, processor_id, file_path, mime_type='application/pdf',
                process_options=None):
        name = self.client.processor_path(self.project_id, self.location, processor_id)
        with open(file_path, 'rb') as f:
            content = f.read()
        request = documentai.ProcessRequest(
            name=name,
            raw_document=documentai.RawDocument(content=content, mime_type=mime_type),
            process_options=process_options)
        return self.client.process_document(request=request).document

    def ingest_for_rag(self, layout_id, file_path, chunk_size=1024):
        options = documentai.ProcessOptions(
            layout_config=documentai.ProcessOptions.LayoutConfig(
                chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
                    chunk_size=chunk_size,
                    include_ancestor_headings=True)))
        doc = self.process(layout_id, file_path, process_options=options)
        batch = self.db.batch()
        for chunk in doc.chunked_document.chunks:
            ref = self.db.collection('rag_chunks').document(chunk.chunk_id)
            batch.set(ref, {
                'content': chunk.content,
                'source': file_path,
                'processed_at': firestore.SERVER_TIMESTAMP})
        batch.commit()
        return len(doc.chunked_document.chunks)

print('DocumentIngester ready')

In [ ]:
# End-to-end test with a research paper
ingester = DocumentIngester(PROJECT_ID, LOCATION)
n = ingester.ingest_for_rag(LAYOUT_ID, 'test.pdf')
print(f'Ingested {n} chunks into Firestore rag_chunks collection')